In [ ]:
suppressPackageStartupMessages({
  library(SingleCellExperiment)
  library(zellkonverter)
  library(ggplot2)
  library(dplyr)
  library(tibble)
  library(Matrix)
  library(S4Vectors)
})

# Load SRTsim
if (requireNamespace("SRTsim", quietly = TRUE)) {
  suppressPackageStartupMessages(library(SRTsim))
} else {
  if (!requireNamespace("devtools", quietly = TRUE)) {
    install.packages("devtools")
  }
  devtools::load_all("SRTsim-main")
}

if (!requireNamespace("pdist", quietly = TRUE)) {
  install.packages("pdist")
}

In [ ]:
h5ad_path <- "lymph_node_annotated.h5ad"
sce <- readH5AD(h5ad_path)

print(dim(sce))
if ("cell_type" %in% colnames(colData(sce))) {
  print(table(colData(sce)$cell_type))
} else {
  stop("Reference data must contain colData(sce)$cell_type")
}

# Choose a count-like assay for SRTsim
assays_avail <- assayNames(sce)
assay_use <- if ("counts" %in% assays_avail) {
  "counts"
} else if ("X" %in% assays_avail) {
  "X"
} else {
  assays_avail[[1]]
}
message("Using assay: ", assay_use)

ref_counts <- assay(sce, assay_use)

# Make sure counts are non-negative integers (SRTsim assumes count models)
ref_counts <- as(ref_counts, "dgCMatrix")
ref_counts@x[ref_counts@x < 0] <- 0
if (!all(abs(ref_counts@x - round(ref_counts@x)) < 1e-6)) {
  message("Assay is not integer; rounding to nearest integer for SRTsim.")
  ref_counts@x <- round(ref_counts@x)
}

# Ensure row/col names exist
if (is.null(rownames(ref_counts))) rownames(ref_counts) <- rownames(sce)
if (is.null(colnames(ref_counts))) colnames(ref_counts) <- colnames(sce)

if ("spatial" %in% reducedDimNames(sce)) {
  xy <- reducedDim(sce, "spatial")
  colData(sce)$x <- xy[, 1]
  colData(sce)$y <- xy[, 2]
  message("Loaded reference (x,y) from reducedDims(sce,'spatial').")
} else if (all(c("x", "y") %in% colnames(colData(sce)))) {
  message("Loaded reference (x,y) from colData(sce).")
} else {
  message("WARNING: Reference has no spatial coords; generating random (x,y) for SRTsim to run.")
  set.seed(123)
  colData(sce)$x <- runif(ncol(sce), min = 0, max = 1)
  colData(sce)$y <- runif(ncol(sce), min = 0, max = 1)
}

ref_locs <- data.frame(
  x = as.numeric(colData(sce)$x),
  y = as.numeric(colData(sce)$y),
  label = as.character(colData(sce)$cell_type),
  row.names = colnames(ref_counts)
)

stopifnot(ncol(ref_counts) == nrow(ref_locs))

In [ ]:
library(ggplot2)

set.seed(123)

# Set the number of cells and square area
n_spots <- 4000
max_dim <- 90

x_coords <- runif(n_spots, 0, max_dim)
y_coords <- runif(n_spots, 0, max_dim)

r <- sqrt(x_coords^2 + y_coords^2)

theta <- atan2(y_coords, x_coords)

spatial_coords <- data.frame(x = x_coords, y = y_coords)

assign_box_niche <- function(r, theta, x, y) {
  niche <- character(length(r))
  
  # Define irregular boundaries (simulating real tissue)
  wave_medulla <- 4 * sin(6 * theta) + rnorm(length(r), 0, 1.5)
  thresh_medulla <- 40 + wave_medulla
  
  # T-Zone outer boundary (r ~ 80): simulating cortex edge
  wave_tzone <- 3 * sin(8 * theta + 2) + rnorm(length(r), 0, 1.5)
  thresh_tzone <- 75 + wave_tzone
  
  # Zone division
  # Medulla
  niche[r < thresh_medulla] <- "Medulla"
  
  # T cell Zone
  is_t_zone <- r >= thresh_medulla & r < thresh_tzone
  niche[is_t_zone] <- "T cell Zone"
  
  # B cell Zone
  niche[r >= thresh_tzone] <- "B cell Zone"
  
  # Germinal Center (GC)
  gc_center_x <- 78
  gc_center_y <- 78
  gc_radius <- 15
  
  dist_to_gc <- sqrt((x - gc_center_x)^2 + (y - gc_center_y)^2)
  gc_noise <- rnorm(length(r), 0, 1.0)
  
  is_b_zone <- niche == "B cell Zone"
  niche[is_b_zone & (dist_to_gc < (gc_radius + gc_noise))] <- "Germinal Center"
  
  niche
}

spatial_coords$niche <- assign_box_niche(r, theta, spatial_coords$x, spatial_coords$y)

spatial_coords$niche <- factor(spatial_coords$niche, 
                               levels = c("Medulla", "T cell Zone", "B cell Zone", "Germinal Center"))

ggplot(spatial_coords, aes(x = x, y = y, color = niche)) +
  geom_point(alpha = 0.6, size = 1.2) +
  theme_minimal() +
  scale_color_manual(values = c(
    "Medulla" = "#F2A994",
    "T cell Zone" = "#39B29A",
    "B cell Zone" = "#F07D6C",
    "Germinal Center" = "#70C8DD"
  )) +
  coord_fixed(ratio = 1) +
  xlim(0, 90) + ylim(0, 90) +
  ggtitle("Simulated Lymph Node Layout (Square Section)")

In [ ]:
# Define Base Probability Matrix 
cell_types <- c(
  "plasma cell", "macrophage", "CD4 T cell", "CD8 T cell", "Naive B cell",
  "Memory B cell", "GC B DZ", "GC B LZ", "T-zone reticular cell",
  "follicular dendritic cell", "pericyte", "interferon-stimulated cell",
  "cDC2", "dendritic cells", "NK cell", "epithelial cell"
)

niches <- c("Medulla", "T cell Zone", "B cell Zone", "Germinal Center")

prob_matrix_base <- matrix(
  0,
  nrow = length(cell_types),
  ncol = length(niches),
  dimnames = list(cell_types, niches)
)

# Medulla (Main: plasma cell, macrophage)
prob_matrix_base["plasma cell", "Medulla"] <- 35.0
prob_matrix_base["macrophage", "Medulla"] <- 20.0
prob_matrix_base["CD4 T cell", "Medulla"] <- 12.0
prob_matrix_base["CD8 T cell", "Medulla"] <- 8.0
prob_matrix_base["Naive B cell", "Medulla"] <- 5.0
prob_matrix_base["Memory B cell", "Medulla"] <- 5.0
prob_matrix_base["T-zone reticular cell", "Medulla"] <- 5.0
prob_matrix_base["pericyte", "Medulla"] <- 3.0
prob_matrix_base["interferon-stimulated cell", "Medulla"] <- 3.0
prob_matrix_base["cDC2", "Medulla"] <- 0.5
prob_matrix_base["dendritic cells", "Medulla"] <- 1.0
prob_matrix_base["NK cell", "Medulla"] <- 2.0

# T cell Zone (Main: CD4 T cell, CD8 T cell)
prob_matrix_base["plasma cell", "T cell Zone"] <- 0.5
prob_matrix_base["macrophage", "T cell Zone"] <- 1.0
prob_matrix_base["CD4 T cell", "T cell Zone"] <- 45.0
prob_matrix_base["CD8 T cell", "T cell Zone"] <- 25.0
prob_matrix_base["Naive B cell", "T cell Zone"] <- 15.0
prob_matrix_base["Memory B cell", "T cell Zone"] <- 1.0
prob_matrix_base["T-zone reticular cell", "T cell Zone"] <- 6.0
prob_matrix_base["pericyte", "T cell Zone"] <- 1.5
prob_matrix_base["interferon-stimulated cell", "T cell Zone"] <- 2.0
prob_matrix_base["cDC2", "T cell Zone"] <- 1.2
prob_matrix_base["dendritic cells", "T cell Zone"] <- 1.0
prob_matrix_base["NK cell", "T cell Zone"] <- 1.0

# B cell Zone (Main: Naive B cell, Memory B cell)
prob_matrix_base["plasma cell", "B cell Zone"] <- 0.1
prob_matrix_base["macrophage", "B cell Zone"] <- 1.5
prob_matrix_base["CD4 T cell", "B cell Zone"] <- 2.0
prob_matrix_base["CD8 T cell", "B cell Zone"] <- 0.5
prob_matrix_base["Naive B cell", "B cell Zone"] <- 65.0
prob_matrix_base["Memory B cell", "B cell Zone"] <- 25.0
prob_matrix_base["T-zone reticular cell", "B cell Zone"] <- 0.1
prob_matrix_base["follicular dendritic cell", "B cell Zone"] <- 5.0
prob_matrix_base["pericyte", "B cell Zone"] <- 0.2
prob_matrix_base["interferon-stimulated cell", "B cell Zone"] <- 0.5
prob_matrix_base["cDC2", "B cell Zone"] <- 0.1
prob_matrix_base["dendritic cells", "B cell Zone"] <- 0.5

# Germinal Center (Main: GC B DZ, GC B LZ)
prob_matrix_base["plasma cell", "Germinal Center"] <- 0.1
prob_matrix_base["macrophage", "Germinal Center"] <- 2.0
prob_matrix_base["CD4 T cell", "Germinal Center"] <- 10.0
prob_matrix_base["CD8 T cell", "Germinal Center"] <- 0.1
prob_matrix_base["Naive B cell", "Germinal Center"] <- 2.0
prob_matrix_base["Memory B cell", "Germinal Center"] <- 0.5
prob_matrix_base["GC B DZ", "Germinal Center"] <- 55.0
prob_matrix_base["GC B LZ", "Germinal Center"] <- 28.0
prob_matrix_base["T-zone reticular cell", "Germinal Center"] <- 0.1
prob_matrix_base["follicular dendritic cell", "Germinal Center"] <- 1.5
prob_matrix_base["pericyte", "Germinal Center"] <- 0.5
prob_matrix_base["interferon-stimulated cell", "Germinal Center"] <- 0.1
prob_matrix_base["cDC2", "Germinal Center"] <- 0.1
prob_matrix_base["dendritic cells", "Germinal Center"] <- 0.1

# Define dominant cell types per Niche
dominant_cells <- list(
  "Medulla" = c("plasma cell", "macrophage"),
  "T cell Zone" = c("CD4 T cell", "CD8 T cell"),
  "B cell Zone" = c("Naive B cell", "Memory B cell"),
  "Germinal Center" = c("GC B DZ", "GC B LZ")
)

# Neighbors based on layout: Medulla <-> T cell Zone <-> B cell Zone (<-> GC inside B)
crosstalk_map <- list(
  "Medulla" = c("T cell Zone"),
  "T cell Zone" = c("Medulla", "B cell Zone"),
  "B cell Zone" = c("T cell Zone", "Germinal Center"),
  "Germinal Center" = c("B cell Zone")
)

# Normalize columns (Base Purity)
prob_matrix_base <- apply(prob_matrix_base, 2, function(x) x / sum(x))
print("Base Probability Matrix:")
print(round(prob_matrix_base, 4))

# Fit SRTsim Model
simSRT_base <- createSRT(count_in = ref_counts, loc_in = ref_locs)

set.seed(1)
message("Fitting SRTsim model (base)... This may take a moment.")
simSRT_base <- srtsim_fit(simSRT_base, marginal = "nb", sim_scheme = "domain")

new_locs_coords <- data.frame(x = spatial_coords$x, y = spatial_coords$y)
old_locs_coords <- as.data.frame(simSRT_base@refcolData[, c("x", "y")])

message("Calculating Nearest Neighbors...")
new_old_dmat <- t(apply(new_locs_coords, 1, function(xy) {
  as.matrix(pdist::pdist(old_locs_coords, xy))
}))
nn_k50 <- t(apply(new_old_dmat, 1, function(d) order(d)[1:50]))

# Simulation Loop
# Purity levels now treated as Scaling Factors for Dominant Cells only
# Factor = 1.0 (Original), < 1.0 (Reduced Purity)
purity_factors <- c(1.0, 0.80, 0.60, 0.40)
names(purity_factors) <- c("100purity", "80purity", "60purity", "40purity")

results_meta <- list()

for (scenario_name in names(purity_factors)) {
  factor_val <- purity_factors[[scenario_name]]
  message(sprintf("\n--- Processing Scenario: %s (Dominant Factor = %.2f with Spatial Crosstalk) ---", scenario_name, factor_val))
  
  prob_curr <- prob_matrix_base
  
  niche_counts <- table(spatial_coords$niche)
  
  for (n in niches) {
    if (factor_val < 1.0) {
      doms <- dominant_cells[[n]]
      idx_dom <- which(rownames(prob_curr) %in% doms)
      
      # Calculate loss
      p_dom_total <- sum(prob_curr[idx_dom, n])
      p_loss <- p_dom_total * (1.0 - factor_val)
      
      # Apply loss to dominant cells
      prob_curr[idx_dom, n] <- prob_curr[idx_dom, n] * factor_val
      
      # Strategy: 70% of loss goes to Crosstalk, 30% to General Noise (background)
      prop_crosstalk <- 0.7
      loss_to_cross <- p_loss * prop_crosstalk
      loss_to_noise <- p_loss * (1 - prop_crosstalk)
      
      neighbor_niches <- crosstalk_map[[n]]
      
      neighbor_total_count <- sum(niche_counts[neighbor_niches])
      
      for (nn in neighbor_niches) {
        # Proportion of crosstalk assigned to this neighbor nn
        prop_nn <- niche_counts[[nn]] / neighbor_total_count
        loss_nn <- loss_to_cross * prop_nn
        
        doms_nn <- dominant_cells[[nn]]
        
        target_cells <- setdiff(doms_nn, doms)
        idx_target <- which(rownames(prob_curr) %in% target_cells)
        
        if (length(idx_target) > 0) {
           w_target <- prob_matrix_base[idx_target, nn]
           
           if (sum(w_target) > 0) {
              prob_curr[idx_target, n] <- prob_curr[idx_target, n] + (w_target / sum(w_target)) * loss_nn
           } else {
              prob_curr[idx_target, n] <- prob_curr[idx_target, n] + (loss_nn / length(idx_target))
           }
        }
      }
      
      # Distribute remainder to Noise (other cells)
      all_neighbor_doms <- unique(unlist(lapply(neighbor_niches, function(nn) dominant_cells[[nn]])))
      idx_others <- which(!rownames(prob_curr) %in% c(doms, all_neighbor_doms))
      
      w_noise <- prob_curr[idx_others, n]
      if (sum(w_noise) == 0) w_noise[] <- 1.0
      prob_curr[idx_others, n] <- prob_curr[idx_others, n] + (w_noise / sum(w_noise)) * loss_to_noise
    }
  }
  
  prob_curr <- apply(prob_curr, 2, function(x) x / sum(x))
  
  message(sprintf("Cell Type Proportions per Niche for %s:", scenario_name))
  print(round(prob_curr, 4))
  
  # Assign Cell Types
  sc_curr <- spatial_coords
  sc_curr$cell_type <- NA
  
  for (n in niches) {
    idx <- which(sc_curr$niche == n)
    if (length(idx) > 0) {
      sc_curr$cell_type[idx] <- sample(
        x = rownames(prob_curr),
        size = length(idx),
        replace = TRUE,
        prob = prob_curr[, n]
      )
    }
  }
  sc_curr$cell_type <- factor(sc_curr$cell_type, levels = rownames(prob_curr))
  
  # Setup Layout in simSRT
  new_locs_df <- data.frame(
    x = sc_curr$x,
    y = sc_curr$y,
    label = as.character(sc_curr$cell_type)
  )
  rownames(new_locs_df) <- paste0("loc", seq_len(nrow(new_locs_df)))
  
  simSRT_curr <- simSRT_base
  simSRT_curr@metaParam$simLocParam <- S4Vectors::SimpleList(
    voting_nn = NA_integer_,
    loc_lay_out = "custom",
    new_loc_num = nrow(new_locs_df),
    simref_nmat = nn_k50
  )
  simSRT_curr@simcolData <- as(new_locs_df, "DFrame")
  
  # Simulate Counts
  message("Simulating counts...")
  simSRT_curr <- srtsim_count(simSRT_curr, nn_num = 5, nn_func = "mean", numCores = 1, verbose = FALSE)
  
  # Export h5ad
  sim_counts <- simCounts(simSRT_curr)
  sim_sce <- SingleCellExperiment(
    assays = list(counts = sim_counts),
    colData = DataFrame(
      x = sc_curr$x,
      y = sc_curr$y,
      niche = sc_curr$niche,
      cell_type = sc_curr$cell_type,
      row.names = colnames(sim_counts)
    )
  )
  reducedDim(sim_sce, "spatial") <- cbind(x = sc_curr$x, y = sc_curr$y)
  
  fname <- sprintf("simulated_lymph_node_SRTsim_%s.h5ad", scenario_name)
  writeH5AD(sim_sce, fname, X_name = "counts")
  message(paste("Saved:", fname))
  
  results_meta[[scenario_name]] <- list(factor = factor_val, file = fname)
}

message("All simulations completed.")


In [ ]:
# spatial visualization for marker gene
plot_gene <- function(sce_obj, gene) {
  if (!(gene %in% rownames(sce_obj))) {
    message("Gene not found: ", gene)
    return(invisible(NULL))
  }
  df <- as.data.frame(colData(sce_obj))
  df$expr <- log1p(as.numeric(counts(sce_obj)[gene, ]))
  ggplot(df, aes(x = x, y = y, color = expr)) +
    geom_point(size = 0.6) +
    scale_color_viridis_c() +
    theme_minimal() +
    coord_fixed() +
    ggtitle(paste0("SRTsim spatial expression: ", gene))
}

plot_gene(sim_sce, "MS4A1")